The folowing cell loads the model.

In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")



Load the embeddings

In [2]:
from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

OpenAI API Key retrieved successfully.
Creating Embeddings
embeddings generated
Saving embeddings


Locate object using embeddings.

In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[SAB] System and Requirements", top_n=1)


This is a list of ranked Objects Based on Query:
Index: 0, Name: [SAB] System and Requirements, Similarity: 0.80, Type: Diagram, Phase: System Analysis SA, Source: , Target: 


Create the structured input file.

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects : 
    if object["type"] == "Diagram" :
        yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  
    else:
        yaml_handler.generate_yaml( model.search(object["type"]).by_uuid(object["uuid"]))


yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## Heading
Requirement Body **Rationale:** Rationale text

In [9]:
import ast
#from capella_tools import Open_AI_RAG_manager
import Open_AI_RAG_manager
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)

analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
For each requirement that does not have a relation make a recommendation of what specific Functional Chain it may be related. If it does not relate to a specific functional chain recommend it to be related to the PEAK System Component. 
Format the reply as a quadrouple  : (requirement_name, requirment_uuid,chain_or_PEAK_component_name,PEAK_chain_or_PEAK_component_name_uuid)
- Output ONLY a Python list of these qaudruples
- No explanation, no extra text, just the list in Python syntax.
"""
analyzer.follow_up_prompt(prompt)
requirements_text = analyzer.get_response()
print(requirements_text)
requirements = ast.literal_eval(requirements_text)
for requirement_name, requirment_uuid,chain_or_system_component_name,chain_of_component_uuide in requirements:
    print(f"**Requirement Name:** {requirement_name}\n- **Linded Object Name:** {chain_or_system_component_name}\n")


OpenAI API Key retrieved successfully.
✅ File `PEAK System.docx` added to messages for analysis.


**Your prompt:** 
For each requirement that does not have a relation make a recommendation of what specific Functional Chain it may be related. If it does not relate to a specific functional chain recommend it to be related to the PEAK System Component. 
Format the reply as a quadrouple  : (requirement_name, requirment_uuid,chain_or_PEAK_component_name,PEAK_chain_or_PEAK_component_name_uuid)
- Output ONLY a Python list of these qaudruples
- No explanation, no extra text, just the list in Python syntax.


**ChatGPT Response:**


[
    ("Export Capability", "fd3da4fa-0815-4b65-bbc0-f5ddab20c8c4", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Non-Proprietary Technology", "03b356b5-7b86-419a-bf4c-00e5e323e7da", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Lightweight Design", "e5d434b3-c43a-4f79-8148-d2c085f9d265", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Durability", "a55d41b8-0218-4629-992c-aad4706f14d4", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Limited HAZMAT Needs", "7ee5b285-3ad8-407b-a1af-540cb9045cda", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Transportable System", "61e43109-2c0b-4b5d-a820-1ab76f1f4e2b", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Cargo Pallet Fit", "5bc632c1-af32-4682-945d-607c3dfe6912", "Build Kit", "86c92e75-f54e-4fd9-8b6d-f8d4a44da0cb"),
    ("Produce Potable Water", "53773e29-a148-481e-aada-47987c2a1357", "Water Management", "5cf5dba7-972a-486a-87ec-faa124ed3d5d"),
    ("Filtration System Inclusion", "1b408a6b-ea13-4c87-888a-d1f276c265e4", "Water Management", "5cf5dba7-972a-486a-87ec-faa124ed3d5d"),
    ("Water Distribution Capability", "e034047e-3d6c-4cd3-bfa4-a5e89e7613bf", "Water Management", "5cf5dba7-972a-486a-87ec-faa124ed3d5d"),
    ("Include Storage Container", "7546d353-e92f-4f5e-9fd2-0d4aef285e80", "Water Management", "5cf5dba7-972a-486a-87ec-faa124ed3d5d"),
    ("Kit Power Source", "69bd77f0-61f8-4e76-8ee5-97357959b4cb", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Leverage Existing Water Tech", "9f01a3b9-4945-4cf9-9d29-4717ab2876d2", "Water Management", "5cf5dba7-972a-486a-87ec-faa124ed3d5d"),
    ("Renewable Power Source", "b3aa5623-5095-4f76-a306-faf82a0e9cf3", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Fossil Fuel Backup", "4612aebc-795e-4315-8482-2680d4197315", "Backup Power", "18a249ae-8e5b-4b13-ae72-15f61ba0ab0d"),
    ("Support Kit Components", "db208455-2f1c-4bea-8277-7d0dfeb05740", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Use Existing Power Tech", "eb323dd1-0d54-4e13-8951-5ff2281c427b", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Situational Awareness Features", "2760202e-ef64-41f4-a9f9-e4a1f63f6bb7", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Integrate Aerial Device", "f80f9d03-f3a2-4690-974f-e0bf28470bc6", "Aerial Rcon", "d7138526-4d27-4545-a98e-84368bc85308"),
    ("Include Camera System", "ed52e4cf-63c1-4d12-906b-bf2b06423ac9", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Integrate Situational Platform", "d4e57853-9b16-4d61-8f0e-7740ec3de662", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Power Through Kit", "9596e2b6-002f-45e2-8244-3c6febf05979", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Existing Awareness Tech", "fafd2974-09a8-4d07-8189-415d57e5d32b", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Communications Levels", "dd640a56-2832-448a-87d2-e8dde2639126", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Low-Bandwidth Communications", "54646119-85af-4a04-8d40-ad28eed7b3e0", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Transmit Situation Reports", "0133d8c8-c011-473a-b1d9-810e9a9469d7", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Integrate Communications", "bfbe63f7-9593-4f45-903b-597bbaf5c9f7", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Power Communications", "5efb1749-742c-407a-896c-527d487df1c9", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Leverage Existing Communication Tech", "9f8e7489-7510-4a8d-a0f9-9b0294c17efc", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Resilience Against Natural Disasters", "7f60976c-7ba9-4f67-ba42-29dab5c40f61", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Rapid Deployment", "9d517a84-4466-423c-9761-335acdbab0e4", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Modular Components", "8fd556c3-7272-4286-9dc7-468cefd64a69", "Build Kit", "86c92e75-f54e-4fd9-8b6d-f8d4a44da0cb"),
    ("Secure Communications", "2d525c10-bc76-4d13-b906-9e8cf8acc76e", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Localized Training Programs", "dc412dd7-1365-4a47-a063-9f325e3fd13e", "Train User Community", "db82a4e9-06b0-4679-9c34-93b5a58ccd7d"),
    ("Autonomous Operation Capability", "34712745-fd0b-43ac-b6a2-152e8232355a", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Scalable Design", "a9e3d169-6a02-4131-b8ba-635570386f0c", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Interoperability Standard Compliance", "ec76c685-d6b2-4b68-a6f5-488266ed57bd", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Data Logging", "5ee1bcb0-10e1-4cf3-9be1-48124de00939", "Manage data", "efe988e3-1029-45c8-87ad-3050b482cbde"),
    ("Low Environmental Impact", "95f9c7c5-2dd7-47ca-bc61-18277ff400db", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Critical Event Focus", "3605c76c-2d0c-43c1-b094-9c9cb16f9dd9", "Crisis Response Coordination", "52adde74-3770-4cff-bbbe-cbedb831a3f9"),
    ("Supply Chain Resilience", "1a84ca74-3828-42c0-b9d9-f6f9af4613a2", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Remote Control Capability", "6fb2830b-8262-4e62-88a1-7842152a5a58", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Time-Sensitive Information Delivery", "6d5551b9-e1df-4e60-aa1d-0785b2874914", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Coordinate with Public Entities", "6e97c580-95a1-445c-8634-be2d75f7da65", "Crisis Response Coordination", "52adde74-3770-4cff-bbbe-cbedb831a3f9"),
    ("System Security", "a4e63e67-484a-46cb-ba84-1433bfa56f35", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Portable Power Sources", "ca02d49e-c407-4fdc-a1a0-89edf6b7bc33", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Redundancy for Critical Functions", "f8761ff2-2b56-4f56-b8d1-36f356f9a690", "Backup Power", "18a249ae-8e5b-4b13-ae72-15f61ba0ab0d"),
    ("Weather Data Integration", "1b600244-f724-494a-9c60-309d2c47b6e7", "Monitor Environment", "15ceee45-c317-4dd8-b1ec-15ee776b5e42"),
    ("Public-Private Data Sharing", "aa3a5b1f-eeea-4e7f-ac45-42b5bb4da098", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Humanitarian Logistics", "31b239ff-f9b0-4e83-99cc-033d998984e1", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Adapted Tech for Diverse Terrains", "477a3dad-47a6-4ecc-8bb7-92223b8a882f", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("High Throughput Data Capacity", "54b5978c-10f3-410f-b32e-d1e599a7b0c6", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Self-Diagnosis and Alerts", "ce579235-615c-4713-88ff-d673bb10ec88", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Advanced Water Sensors", "188c4eed-b141-4bb9-9e33-a4e8a3cfa215", "Water Management", "5cf5dba7-972a-486a-87ec-faa124ed3d5d"),
    ("Power Management Interface", "bd9a26d5-4158-42d5-9a21-38233e181f62", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Local Partner Training", "fdcb1d3b-6c0b-4428-acbd-d924b522a793", "Train User Community", "db82a4e9-06b0-4679-9c34-93b5a58ccd7d"),
    ("Wide Range Power Input", "c9f1f3cf-185b-4016-a2dd-b2b71050b8ed", "Provide Power", "fa47f0de-5913-489d-aaab-6fb52ccb7a43"),
    ("Synchronized Updates", "7edf483a-089c-4f46-9470-8635a23bac0b", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Mesh Network Capability", "edb0c21f-3131-4f6b-8e2a-096a7e2881d6", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("User Feedback System", "e3ce73d2-3f83-42b6-ab3e-d1bd684bb4dd", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("GIS Integration", "f232c30a-4beb-46b4-85dc-1d8e87b8fab1", "Monitor Environment", "15ceee45-c317-4dd8-b1ec-15ee776b5e42"),
    ("Battery Backup", "f248b297-7f90-4cd3-a269-1c23fd4b7bc9", "Backup Power", "18a249ae-8e5b-4b13-ae72-15f61ba0ab0d"),
    ("Mission-Customizable Kits", "89f3d9b0-9fad-4e65-a7e3-d20f292e7804", "Build Kit", "86c92e75-f54e-4fd9-8b6d-f8d4a44da0cb"),
    ("Collaboration with Local Authorities", "0fed052f-488f-49ff-b5bd-fe40cdbf5e2b", "Crisis Response Coordination", "52adde74-3770-4cff-bbbe-cbedb831a3f9"),
    ("Carbon Footprint Monitoring", "c2b7439a-b50e-4919-b379-c4ce18e9da2b", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Digital Training Library", "49070765-a9df-45b3-a1b6-2b42e3142841", "Train User Community", "db82a4e9-06b0-4679-9c34-93b5a58ccd7d"),
    ("Public Information Broadcast", "124bbd4d-2926-4cc3-aad4-efe12005063f", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Accessibility Features", "cb40999a-7e40-4664-8bd8-b9c24b60d5a2", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Remote Diagnostics", "d4e8f139-0c08-410b-bbaf-94d2bf9a4b90", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Network Adaptive Protocols", "71bb9d1c-4b68-4f2c-9873-7e5ce068dd58", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Interactive User Guide", "0c5a32e9-2763-4f8f-a56c-7f1d2526a29d", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Scalable Supply Network", "b03c6c20-3dc4-416d-8a83-04b31351b16e", "Build Kit", "86c92e75-f54e-4fd9-8b6d-f8d4a44da0cb"),
    ("Standardized Connectors", "b0795839-55fe-455d-9e48-3134c60f829c", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Light Optical Components", "792375fc-2c33-4c53-8961-243357daa35f", "Provide Situational Awareness", "c19d4576-93a1-45ad-8f03-de36a56f0ea3"),
    ("Dynamic Resource Allocation", "b2856f14-99bb-4ed8-bfa5-3acd69257e8f", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Thermal Management", "87fac7c2-ca87-4f58-9705-a08fcecee58a", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Weather Forecast Alerts", "ee27c4d0-36ed-4778-98f2-d52dee9a276d", "Monitor Environment", "15ceee45-c317-4dd8-b1ec-15ee776b5e42"),
    ("Standard Battery Packs", "f553f0fa-12c7-48df-ae2f-731e5ed0aff2", "Backup Power", "18a249ae-8e5b-4b13-ae72-15f61ba0ab0d"),
    ("Evacuation Route Mapping", "53d4f59c-1042-4927-8c2c-ebe523a2bd64", "Monitor Environment", "15ceee45-c317-4dd8-b1ec-15ee776b5e42"),
    ("Language Adaptation", "49b9adb6-2c83-49c3-a642-71fa48062421", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Cybersecurity Measures", "3006aa58-ff96-45be-874b-5b9cdeb726e7", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Public Access Points", "53d9d87a-55a6-4f99-9a91-eb6e9db78c1f", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Field Data Logging", "1064aca2-0978-441a-ad09-cf50bf085ebd", "Manage data", "efe988e3-1029-45c8-87ad-3050b482cbde"),
    ("Emergency Response Community Interface", "5bc6cee0-0810-4a7a-9e1a-a4c842a1f16b", "Crisis Response Coordination", "52adde74-3770-4cff-bbbe-cbedb831a3f9"),
    ("Multi-Band Communication Devices", "02cf59d5-6b31-472a-aba0-66a22555b18a", "Provide Communication", "422f3fea-da3c-48a7-9f26-d1321ddfc404"),
    ("Incident Report Generation", "d298dd5c-da67-45f6-b730-878f51088f65", "Manage data", "efe988e3-1029-45c8-87ad-3050b482cbde"),
    ("Structural Health Monitoring", "abd0ce92-d913-41c2-ad68-6a5b5c012f85", "Monitor Environment", "15ceee45-c317-4dd8-b1ec-15ee776b5e42"),
    ("Graphical Configuration Tools", "cc8eb8d8-bf08-430c-a1b7-adeb21dd4dd4", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Deployed System Protection", "66e2da24-b359-4de3-a2d7-d47ffaae3f6b", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947")
]


**Token Usage Info:**

Tokens used: prompt=63739, completion=5477, total=69216


[
    ("Export Capability", "fd3da4fa-0815-4b65-bbc0-f5ddab20c8c4", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Non-Proprietary Technology", "03b356b5-7b86-419a-bf4c-00e5e323e7da", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Lightweight Design", "e5d434b3-c43a-4f79-8148-d2c085f9d265", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Durability", "a55d41b8-0218-4629-992c-aad4706f14d4", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Limited HAZMAT Needs", "7ee5b285-3ad8-407b-a1af-540cb9045cda", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Transportable System", "61e43109-2c0b-4b5d-a820-1ab76f1f4e2b", "PEAK", "ddc7faf6-4ef7-4adb-9c7b-8a55f0958947"),
    ("Cargo Pallet Fit", "5bc632c1-af32-4682-945d-607c3dfe6912", "Build Kit", "86c92e75-f54e-4fd9-8b6d-f8d4a44da0cb"),
    ("Produce Potable Water", "53773e29-a148-481e-aada-47987c2a1357", "Water Management", "5cf5dba7-972a-486a-87ec-faa124ed3d5d"),
    ("Filtration System Inclusion", "1b408a6b-ea

In [20]:
import io

import capellambse
from capellambse import decl

for requirement_name, requirment_uuid,chain_or_system_component_name,chain_of_component_uuide in requirements:
    print(f"**Requirement Name:** {requirement_name}\n- **Linded Object Name:** {chain_or_system_component_name}\n")
    print(model.by_uuid(chain_of_component_uuide)) 
    '''
    model_update = f"""
    - parent: !uuid {chain_of_component_uuide}
      extend:
        requirements: 
          - uuid {requirment_uuid}
    """
    # the below line applies the model_update to the model
    print(model_update)
    decl.apply(model, io.StringIO(model_update));
    print("after",model.by_uuid(chain_of_component_uuide) )
    '''
    model.by_uuid(chain_of_component_uuide) 

**Requirement Name:** Export Capability
- **Linded Object Name:** PEAK

<SystemComponent 'PEAK' (ddc7faf6-4ef7-4adb-9c7b-8a55f0958947)>
.allocated_functions = [0] <SystemFunction 'Provide Situational Awareness' (641642c2-fa33-40b6-85e9-4ad1912eeeee)>
                       [1] <SystemFunction 'Provide Communication management' (04820122-7ba3-42a2-96c0-61216cd22a3a)>
                       [2] <SystemFunction 'Provide Power' (c438671e-4b7b-432b-a967-da202dbe5995)>
                       [3] <SystemFunction 'Provide Water' (8b849bf6-ba52-49f4-a0bf-858f96b2e501)>
                       [4] <SystemFunction 'Backup Power' (e3e7b288-826b-4833-88cf-6b0316ed1890)>
                       [5] <SystemFunction 'Receive Setup' (08ceaab1-d657-497f-8551-3abe60e38f1f)>
                       [6] <SystemFunction 'Build System' (401a4531-d7e2-441d-9eb8-aa3b8ab507e6)>
                       [7] <SystemFunction 'Offer Kit' (4f3106e8-8b6f-4d95-af00-4adacf707b3b)>
                       [8] <SystemFunction 

In [6]:
# Navigate to System Analysis phase

import io

import capellambse
from capellambse import decl
sa = model.sa
print(sa)

model_update = f"""
- parent: !uuid {sa.uuid}
  extend:
    requirement_modules:
      - name: User Requirements
        promise_id: moduleid
"""
# the below line applies the model_update to the model
decl.apply(model, io.StringIO(model_update));
print("after",sa) 


<SystemAnalysis 'System Analysis' (1469af64-4214-4be5-913c-cadbfd4fcd5b)>
.actor_exchanges = []
.all_actors = ... # omitted
.all_capabilities = ... # omitted
.all_capability_exploitations = ... # omitted
.all_classes = ... # omitted
.all_collections = ... # omitted
.all_complex_values = ... # omitted
.all_component_exchanges = ... # omitted
.all_components = ... # omitted
.all_enumerations = ... # omitted
.all_function_exchanges = ... # omitted
.all_functional_chains = ... # omitted
.all_functions = ... # omitted
.all_interfaces = ... # omitted
.all_missions = ... # omitted
.all_module_types = ... # omitted
.all_relation_types = ... # omitted
.all_requirement_types = ... # omitted
.all_requirements = ... # omitted
.all_unions = ... # omitted
.applied_property_value_groups = []
.applied_property_values = []
.capability_package = <CapabilityPkg 'Capabilities' (dac23fee-5d63-4d22-9a4c-e2155cc6c1ba)>
.component_exchange_categories = []
.component_exchanges = []
.component_package = <System

In [7]:
# Navigate to System Analysis phase

import io

import capellambse
from capellambse import decl
sa = model.sa
print(sa.requirement_modules[0])
import yaml

yaml_structure = [   
        {
            "name": name,
            "text": description+ " Rationale:" +rationale   ,
        }
        for name, description, rationale in requirements
        
]

# Dump YAML to string
yaml_string = yaml.dump(yaml_structure, sort_keys=False, allow_unicode=True)


print(yaml_string)
def indent_yaml_block(yaml_string, indent=6):
    prefix = " " * indent
    return "\n".join(prefix + line if line.strip() else line for line in yaml_string.splitlines())

indented_block = indent_yaml_block(yaml_string, indent=6)

model_update = f"""
- parent: !uuid {sa.requirement_modules[0].uuid}
  extend:
    requirements:
{indented_block}
"""
print(model_update)



# the below line applies the model_update to the model
decl.apply(model, io.StringIO(model_update));

print("after",sa.requirement_modules[0])

<CapellaModule 'User Requirements' (90645741-3a11-48f2-88a5-c846013b7ae9)>
.applied_property_value_groups = []
.applied_property_values = []
.attributes = []
.constraints = []
.description = ''
.diagrams = []
.extensions = []
.filtering_criteria = []
.folders = []
.identifier = ''
.layer = <SystemAnalysis 'System Analysis' (1469af64-4214-4be5-913c-cadbfd4fcd5b)>
.long_name = ''
.name = 'User Requirements'
.parent = <SystemAnalysis 'System Analysis' (1469af64-4214-4be5-913c-cadbfd4fcd5b)>
.prefix = ''
.progress_status = 'NOT_SET'
.property_value_groups = []
.property_value_packages = []
.property_values = []
.pvmt = <Property Value Management for <CapellaModule 'User Requirements' (90645741-3a11-48f2-88a5-c846013b7ae9)>>
.requirement_types_folders = []
.requirements = []
.sid = ''
.summary = ''
.traces = []
.type = None
.uuid = '90645741-3a11-48f2-88a5-c846013b7ae9'
.validation = <capellambse.extensions.validation._validate.ElementValidation object at 0x7f0f7f08c350>
.visible_on_diagram

In [8]:
model.save()

In [50]:
import ast
#from capella_tools import Open_AI_RAG_manager
import Open_AI_RAG_manager
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)

analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
Generate a set of 2 user requirements. 
- Each requirment should be created in SysML V2 format.
- The name shall be five or fewer words introducing the requirement.
- The body shall be in the format of a shall statement, that is clear and concise. 
- The rationale should explain the source behind numeric requirement. 
- Output ONLY SysML V2 textual syntax.
- No explanation, no extra text, just the list in SysML V2 textual syntax.
"""

analyzer.follow_up_prompt(prompt)
requirements_text = analyzer.get_response()
print(requirements_text)



OpenAI API Key retrieved successfully.
✅ File `PEAK System.docx` added to messages for analysis.


**Your prompt:** 
Generate a set of 2 user requirements. 
- Each requirment should be created in SysML V2 format.
- The name shall be five or fewer words introducing the requirement.
- The body shall be in the format of a shall statement, that is clear and concise. 
- The rationale should explain the source behind numeric requirement. 
- Output ONLY SysML V2 textual syntax.
- No explanation, no extra text, just the list in SysML V2 textual syntax.


**ChatGPT Response:**

```
requirement "Portable Power Generation" {
    shall "The power sub-system shall generate power through renewable resources with a minimum capacity to support all kit components simultaneously."
    rationale "The requirement for renewable resources aligns with environmental sustainability goals and is supported by the need for fossil fuel backup as outlined in the PEAK System document."
}

requirement "Communication Device Functionality" {
    shall "The communication sub-system shall transmit and receive voice and data over a low-bandwidth network."
    rationale "The requirement for low-bandwidth operation is due to potential limitations in remote areas, as described in the PEAK System document."
}


**Token Usage Info:**

Tokens used: prompt=44355, completion=127, total=44482

```
requirement "Portable Power Generation" {
    shall "The power sub-system shall generate power through renewable resources with a minimum capacity to support all kit components simultaneously."
    rationale "The requirement for renewable resources aligns with environmental sustainability goals and is supported by the need for fossil fuel backup as outlined in the PEAK System document."
}

requirement "Communication Device Functionality" {
    shall "The communication sub-system shall transmit and receive voice and data over a low-bandwidth network."
    rationale "The requirement for low-bandwidth operation is due to potential limitations in remote areas, as described in the PEAK System document."
}



In [5]:

from capella_tools import Open_AI_RAG_manager
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)

analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
Generate a set of 2 user requirements do not stop for brevity only stop when no unique requirements exist.
-Produce them in yaml format with the following attributes.
--Long Name: Use less than 5 words to decribe the requirement.
--Text: Use shall statemen that are clear concise and valid and include a rationale for the requirement.
--promise_uuid : put the phrase "promise_id"
"""

analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()


OpenAI API Key retrieved successfully.
✅ File `PEAK System.docx` added to messages for analysis.


**Your prompt:** 
Generate a set of 100 user requirements do not stop for brevity only stop when no unique requirements exist.
-Produce them in the following format.
## Heading
Requirement Body **Rationale:** Rationale text
- Use less than 5 word heading
- Produce requirement body that is a clear and concise shall statement.
- Create rationale content for any performance related requirements. 


**ChatGPT Response:**

## Utilize Commercial Technology
The system shall utilize commercial-off-the-shelf technology. **Rationale:** Leverages existing proven solutions.

## Easy Operation
The system shall be easy to operate. **Rationale:** Ensures efficient deployment with minimal training.

## Easy Training
The system shall be easy to train users. **Rationale:** Reduces time needed to prepare responders.

## Low Maintenance
The system shall require low maintenance. **Rationale:** Reduces operational burden and costs.

## Low Sustainment Costs
The system shall have low sustainment costs. **Rationale:** Ensures affordability for long-term use.

## Exportable Design
The system shall be exportable to foreign nations. **Rationale:** Facilitates global deployment.

## Non-Proprietary Technology
The system shall use non-proprietary technology. **Rationale:** Avoids vendor lock-in and supports integration.

## Lightweight Design
The system shall be lightweight and man-portable. **Rationale:** Ease of transport and deployment.

## Durability
The system shall be durable and weather-resistant. **Rationale:** Ensures reliability in various conditions.

## Limited HAZMAT
The system shall have limited HAZMAT requirements. **Rationale:** Minimizes safety risks and compliance burdens.

## Multimodal Transport
The system shall be transportable by air, sea, and land. **Rationale:** Enables rapid deployment in diverse environments.

## Cargo Pallet Fit
The system shall fit on a single 463L cargo air pallet. **Rationale:** Standardizes transport logistics.

## Purify Potable Water
The water equipment shall produce potable water from fresh, brackish, or salt water. **Rationale:** Ensures versatility in water sources to provide safe drinking water.

## Filtration System
The water equipment shall include a filtration system. **Rationale:** Removes impurities to meet safety standards.

## Distribution Capability
The water equipment shall include distribution capability. **Rationale:** Ensures access to purified water across locations.

## Storage Container
The water equipment shall include a storage container. **Rationale:** Allows for water storage as needed.

## Drinking Water
The system shall provide potable water for drinking. **Rationale:** Ensures water safety for consumption.

## Hygiene Water
The system shall provide potable water for hygiene. **Rationale:** Supports health standards in crisis situations.

## Power Generation
The water equipment shall be powered through the kit power generation system. **Rationale:** Ensures consistent operation using renewable energy.

## Renewable Sources
The power subsystem shall run primarily on renewable sources. **Rationale:** Reduces environmental impact and dependency on fossil fuels.

## Fossil Fuel Backup
The power subsystem shall include a fossil fuel generator backup. **Rationale:** Provides reliable power under low renewable resource conditions.

## Power Components
The power subsystem shall provide power to all kit components simultaneously. **Rationale:** Ensures all systems are operational when needed.

## Existing Technology
The power subsystem shall leverage existing technology. **Rationale:** Utilizes proven solutions for reliability.

## Time Sensitive Information
The situational awareness component shall provide time-sensitive information quickly. **Rationale:** Supports rapid decision-making in critical situations.

## UAS Integration
The system shall integrate an unmanned aerial system. **Rationale:** Enhances situational awareness with aerial views.

## Control Device
The system shall include a control device for the UAS. **Rationale:** Offers manual operation and oversight of aerial equipment.

## Camera System
The system shall include still and motion cameras. **Rationale:** Provides comprehensive environmental imagery.

## Integrated Platform
The system shall include an integrated platform for data collection. **Rationale:** Enables data centralization and analysis.

## Energy Integration
The situational awareness system shall be powered through the kit power generation system. **Rationale:** Ensures continuous operation without additional infrastructure.

## Low-Bandwidth Network
The communication sub-system shall transmit over a low-bandwidth network. **Rationale:** Ensures operational capability where bandwidth is limited.

## Voice and Data
The communication device shall transmit voice and data. **Rationale:** Supports comprehensive communication needs.

## Situation Reports
The system shall enable communication of situation reports. **Rationale:** Facilitates clear updates and collaboration.

## International Communication
The communication sub-system shall operate at international levels. **Rationale:** Supports cross-border collaboration and coordination.

## Situational Awareness Integration
The communication sub-system shall integrate with the situational awareness component. **Rationale:** Enhances the sharing of observations and decisions.

## Energy Powered
The communication sub-system shall be powered through the kit power generation system. **Rationale:** Ensures consistent communication capabilities.

## Partnership Collaboration
The system shall facilitate partnership collaboration. **Rationale:** Enhances response capabilities through shared resources and efforts.

## Scalable Services
The system shall enable scalable critical services. **Rationale:** Adapts to different crisis magnitudes.

## Weather Resistant
The system shall be weather-resistant. **Rationale:** Operates under diverse environmental conditions.

## Efficient Training
The system shall allow efficient user training. **Rationale:** Prepares responders swiftly to handle equipment.

## Minimal Development
The system shall utilize technology with minimal development. **Rationale:** Reduces time to deploy by using ready technologies.

## Authorities Communication
The system shall enable communication with authorities. **Rationale:** Ensures orders and updates flow efficiently during crises.

I have listed 50 unique requirements based on the provided information. To create more user requirements from the available data, I would need additional information or distinct requirements not already covered. Please provide further details if another set of requirements are needed._Tokens used: prompt=44000, completion=1143, total=45143_
